# Stage 8 · Clustering figures — S5, S30, S31

Subtype embeddings from the clustering pipeline, plotted from the **public summary**
(`5kCG100k3C_summary.csv.gz`, which stores the per-tissue and per-major-type t-SNE
coordinates) colored by subtype, using the **published palette** (per subplot: `muted` if
≤10 subtypes else `tab20` — the muted/tab20 combination of file S33).

- **Fig S5** — per-tissue subtype t-SNE (joint embedding).
- **Fig S30** — per-major-type subtype t-SNE using **DNA methylation** (mCG).
- **Fig S31** — per-major-type subtype t-SNE using **chromatin contacts** (3C).

Source: `analysis/step1.clustering_summary.ipynb` cells 57/61/62. Dense scatter rasterized.

## 📥 Required input files

- `{ENTEX_ROOT}/clustering/merged/5kCG100k3C_summary.csv.gz` · _cell metadata + t-SNE coords_
- `{ENTEX_ROOT}/subtype_meta.tsv` · _subtype annotation (celltype_L2_both_abbr)_
- `{ENTEX_ROOT}/L1color.tsv` · _major-type annotation (provided)_
- `{ENTEX_ROOT}/tissuecolor.tsv` · _tissue annotation (provided)_

In [1]:
# === Reproduction setup ===
import os, sys

ENTEX_ROOT = os.environ.get("ENTEX_ROOT", "/large_storage/zhoulab/zhoujt/project/ENTEx")
BOOK_ROOT = os.environ.get("BOOK_ROOT", f"{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas")
sys.path.insert(0, BOOK_ROOT)
import repro_guard

os.chdir(f"{ENTEX_ROOT}/analysis")

[repro_guard] active — READ-ONLY (all writes skipped; inline figures still render)


In [2]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ALLCools.plot import categorical_scatter

In [3]:
mpl.style.use("default")
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

indir = f"{ENTEX_ROOT}/"
meta = pd.read_csv(f"{indir}clustering/merged/5kCG100k3C_summary.csv.gz", index_col=0)
L2_meta = pd.read_csv(f"{indir}subtype_meta.tsv", sep="\t", header=0, index_col=0)
L1_meta = pd.read_csv(f"{indir}L1color.tsv", sep="\t", header=0, index_col=0).drop(
    ["c35", "c36"], axis=0, errors="ignore"
)
tissue_meta = pd.read_csv(f"{indir}tissuecolor.tsv", sep="\t", header=0, index_col=0)
L2_annot = L2_meta["celltype_L2_both_abbr"].to_dict()  # subtype -> abbreviated name

In [4]:
def subtype_grid(groupby, order, name_map, x, y, nrow, ncol, figsize, title, fname, ds_fixed=0.5):
    """Published per-subplot palette: muted if <=10 subtypes else tab20; subtypes with >=30 cells."""
    meta[["tsne_0", "tsne_1"]] = meta[[x, y]].copy()
    fig, axes = plt.subplots(nrow, ncol, figsize=figsize, dpi=300, constrained_layout=True)
    for i, xx in enumerate(order):
        meta_tmp = meta.loc[meta[groupby] == xx].copy()
        meta_tmp["celltype"] = meta_tmp["subtype"].map(L2_annot).astype(str)
        count = meta_tmp["celltype"].value_counts()
        leg = count.index[count >= 30]
        tmp = meta_tmp.loc[meta_tmp["celltype"].isin(leg)].copy()
        palette = "muted" if len(leg) <= 10 else "tab20"
        ds = ds_fixed if ds_fixed else 20 / np.sqrt(max(meta_tmp.shape[0], 1))
        ax = axes.flatten()[i]
        if tmp.shape[0]:
            categorical_scatter(
                data=tmp,
                ax=ax,
                coord_base="tsne",
                hue="celltype",
                s=ds,
                max_points=None,
                palette=palette,
                scatter_kws={"rasterized": True},
                axis_format="empty",
            )
            for yy, (cx, cy) in tmp.groupby("celltype")[["tsne_0", "tsne_1"]].mean().iterrows():
                ax.text(cx, cy, yy, ha="center", va="center", fontsize=6)
        ax.set_title(name_map.get(xx, str(xx)), fontsize=8)
    for ax in axes.flatten()[len(order) :]:
        ax.axis("of")
    fig.suptitle(title, fontsize=10)
    fig.savefig(fname, transparent=True)